# Ticker Price Targets vs Price From Financial Modeling Prep

This notebook pulls the latest consensus price target snapshot for any ticker from FMP's `price-target-consensus` endpoint, then reconstructs a historical target band from FMP's analyst target history so we can compare price target trends against the stock's price.

Change the local `TICKER` parameter near the top of this notebook and rerun it. Inputs like `BRK\\B`, `BRK/B`, and `BRK.B` are normalized automatically for FMP.

Note: FMP's stable `price-target-consensus` endpoint returns the current snapshot only. The historical high, low, median, and consensus series below are reconstructed from FMP's `price-target-news` history using a trailing 365-day window of analyst targets.

In [ ]:
# 2. Set notebook parameters and imports
from pathlib import Path
import os

import pandas as pd
import requests

price_target_params = {
    "ticker_str": "SOXL",
    "rolling_window_days": 365,
}

TICKER = price_target_params["ticker_str"]
ROLLING_WINDOW_DAYS = price_target_params["rolling_window_days"]

price_target_params

In [ ]:
from pathlib import Path
import sys

BASE_URL = "https://financialmodelingprep.com/stable"
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()

from Quantapp.secrets import load_project_env

def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def fetch_all_price_target_news(symbol: str, page_size: int = 100, max_pages: int = 100) -> pd.DataFrame:
    pages = []

    for page in range(max_pages):
        payload = request_json("price-target-news", symbol=symbol, limit=page_size, page=page)
        page_frame = pd.DataFrame(payload)
        if page_frame.empty:
            break
        pages.append(page_frame)
        if len(page_frame) < page_size:
            break

    if not pages:
        raise RuntimeError(f"FMP did not return any price-target news rows for {symbol}.")

    target_news = pd.concat(pages, ignore_index=True)
    target_news = target_news.drop_duplicates().copy()

    if "publishedDate" not in target_news.columns:
        raise RuntimeError(f"FMP price-target-news response for {symbol} is missing publishedDate.")

    target_news["targetDate"] = pd.to_datetime(
        target_news["publishedDate"], errors="coerce", utc=True
    ).dt.tz_convert(None).dt.normalize()
    target_news["priceTarget"] = pd.to_numeric(target_news.get("priceTarget"), errors="coerce")
    target_news["adjPriceTarget"] = pd.to_numeric(
        target_news.get("adjPriceTarget", target_news["priceTarget"]),
        errors="coerce",
    )
    target_news["priceWhenPosted"] = pd.to_numeric(target_news.get("priceWhenPosted"), errors="coerce")
    target_news["effectiveTarget"] = target_news["adjPriceTarget"].where(
        target_news["adjPriceTarget"].notna(),
        target_news["priceTarget"],
    )

    target_news = target_news.dropna(subset=["targetDate", "effectiveTarget"]).sort_values(
        ["targetDate", "effectiveTarget", "newsTitle"],
        kind="mergesort",
    ).reset_index(drop=True)

    if target_news.empty:
        raise RuntimeError(f"FMP returned price-target news for {symbol}, but none contained usable targets.")

    return target_news


def build_historical_target_band(
    price_history: pd.DataFrame,
    target_news: pd.DataFrame,
    rolling_window_days: int,
) -> pd.DataFrame:
    if price_history.empty:
        return pd.DataFrame(
            columns=["date", "targetLow", "targetMedian", "targetConsensus", "targetHigh", "targetCount"]
        )

    history = price_history.loc[:, ["date"]].copy()
    history["date"] = pd.to_datetime(history["date"], errors="coerce")
    history = history.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")

    targets = target_news.loc[:, ["targetDate", "effectiveTarget"]].copy()
    targets["targetDate"] = pd.to_datetime(targets["targetDate"], errors="coerce")
    targets["effectiveTarget"] = pd.to_numeric(targets["effectiveTarget"], errors="coerce")
    targets = targets.dropna(subset=["targetDate", "effectiveTarget"]).sort_values("targetDate")

    band_rows = []
    rolling_window = pd.Timedelta(days=rolling_window_days)

    for current_date in history["date"]:
        window_start = current_date - rolling_window
        active_targets = targets.loc[
            (targets["targetDate"] >= window_start) & (targets["targetDate"] <= current_date),
            "effectiveTarget",
        ]
        if active_targets.empty:
            continue

        band_rows.append(
            {
                "date": current_date,
                "targetLow": active_targets.min(),
                "targetMedian": active_targets.median(),
                "targetConsensus": active_targets.mean(),
                "targetHigh": active_targets.max(),
                "targetCount": int(active_targets.count()),
            }
        )

    return pd.DataFrame(band_rows)


SYMBOL = normalize_symbol(TICKER)
SYMBOL

In [ ]:
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

consensus_snapshot = pd.DataFrame(request_json("price-target-consensus", symbol=SYMBOL))
target_news = fetch_all_price_target_news(SYMBOL)
price_history = pd.DataFrame(request_json("historical-price-eod/light", symbol=SYMBOL))

price_history["date"] = pd.to_datetime(price_history["date"])
price_history["price"] = pd.to_numeric(price_history["price"], errors="coerce")
price_history = (
    price_history.loc[price_history["date"] >= target_news["targetDate"].min(), ["date", "price"]]
    .sort_values("date")
    .reset_index(drop=True)
)

consensus_snapshot


In [ ]:
target_band = build_historical_target_band(price_history, target_news, ROLLING_WINDOW_DAYS)
plot_df = price_history.merge(target_band, on="date", how="left")

latest_reconstructed = plot_df.dropna(subset=["targetConsensus"]).iloc[-1]
latest_snapshot = consensus_snapshot.iloc[0]

comparison = pd.DataFrame(
    [
        {
            "series": f"Trailing {ROLLING_WINDOW_DAYS}-day reconstructed band",
            "date": latest_reconstructed["date"].date(),
            "targetLow": round(latest_reconstructed["targetLow"], 2),
            "targetMedian": round(latest_reconstructed["targetMedian"], 2),
            "targetConsensus": round(latest_reconstructed["targetConsensus"], 2),
            "targetHigh": round(latest_reconstructed["targetHigh"], 2),
            "targetCount": int(latest_reconstructed["targetCount"]),
        },
        {
            "series": "Latest FMP price-target-consensus snapshot",
            "date": pd.Timestamp.today().date(),
            "targetLow": latest_snapshot["targetLow"],
            "targetMedian": latest_snapshot["targetMedian"],
            "targetConsensus": latest_snapshot["targetConsensus"],
            "targetHigh": latest_snapshot["targetHigh"],
            "targetCount": None,
        },
    ]
)

comparison


In [ ]:
latest_market_price = (
    float(quote_snapshot[0].get("price"))
    if quote_snapshot and quote_snapshot[0].get("price") is not None
    else float(plot_df["price"].dropna().iloc[-1])
)

series_targets = [
    {
        "series": f"Trailing {ROLLING_WINDOW_DAYS}-day reconstructed band",
        "referenceDate": latest_reconstructed["date"].date(),
        "referencePrice": float(latest_reconstructed["price"]),
        "targetLow": latest_reconstructed["targetLow"],
        "targetMedian": latest_reconstructed["targetMedian"],
        "targetConsensus": latest_reconstructed["targetConsensus"],
        "targetHigh": latest_reconstructed["targetHigh"],
    },
    {
        "series": "Latest FMP price-target-consensus snapshot",
        "referenceDate": pd.Timestamp.today().date(),
        "referencePrice": latest_market_price,
        "targetLow": latest_snapshot["targetLow"],
        "targetMedian": latest_snapshot["targetMedian"],
        "targetConsensus": latest_snapshot["targetConsensus"],
        "targetHigh": latest_snapshot["targetHigh"],
    },
]

target_label_map = {
    "targetLow": "Low target",
    "targetMedian": "Median target",
    "targetConsensus": "Consensus target",
    "targetHigh": "High target",
}

target_position_rows = []
for target_set in series_targets:
    reference_price = target_set["referencePrice"]

    for key, label in target_label_map.items():
        target_value = pd.to_numeric(target_set[key], errors="coerce")
        if pd.isna(target_value) or target_value == 0 or reference_price == 0:
            continue

        pct_above_below_target = ((reference_price / target_value) - 1) * 100
        pct_move_to_target = ((target_value / reference_price) - 1) * 100

        target_position_rows.append(
            {
                "series": target_set["series"],
                "referenceDate": target_set["referenceDate"],
                "referencePrice": round(reference_price, 2),
                "targetLine": label,
                "targetValue": round(float(target_value), 2),
                "position": "Above" if pct_above_below_target > 0 else "Below" if pct_above_below_target < 0 else "At",
                "pctAboveBelowTarget": round(pct_above_below_target, 2),
                "pctMoveToTarget": round(pct_move_to_target, 2),
            }
        )

target_position_summary = pd.DataFrame(target_position_rows)
target_position_summary

In [ ]:
# 7. Plot analyst price target band
from Quantapp.visualization.views.single_asset_profile.valuation.fair_value import plot_analyst_price_target_band

fig = plot_analyst_price_target_band(
    plot_df,
    latest_snapshot,
    symbol=SYMBOL,
    chart_label=chart_label,
    rolling_window_days=ROLLING_WINDOW_DAYS,
)
fig.show(config={"responsive": True, "displaylogo": False})


In [ ]:
# 8. Plot price premium/discount to analyst targets
from Quantapp.visualization.views.single_asset_profile.valuation.fair_value import plot_price_target_premium_discount

relative_fig = plot_price_target_premium_discount(plot_df, chart_label=chart_label)
relative_fig.show(config={"responsive": True, "displaylogo": False})
